# Skadi BI — Silver do App

Camada Silver única e compartilhada pelos painéis do Gestor e do Funcionário. Ela padroniza os snapshots Bronze por assunto, preserva a granularidade original e publica o estado atual de cada câmara para facilitar o consumo na Gold.

## Escopo desta versão

- Corrige tipos, remove espaços e padroniza códigos categóricos em maiúsculas.
- Remove duplicidades pela chave primária da origem e valida chaves, faixas de temperatura e relacionamentos.
- Preserva os valores de `nivel_gravidade` sem agrupá-los ou validar uma lista fixa, até a confirmação do grupo.
- Mantém separados o estado térmico/conectividade da câmara e o estado operacional do alerta.
- Calcula `OFFLINE` após 10 minutos sem nova leitura, considerando dois ciclos de atualização de 5 minutos. `SEM_DADOS` identifica câmaras sem histórico. Essa classificação fica na Silver; sua exibição como KPI na Gold continua opcional.
- Não calcula `sla_vencido` nem prazos de reconhecimento/resolução. Os limites de escalonamento de 40% e 70% permanecem como dados/regra operacional até a definição final do SLA pelo grupo.

A Silver não agrega KPIs por painel. Os marts Gold do Gestor e do Funcionário devem agregar estas tabelas conforme suas próprias perguntas.


In [ ]:
import re

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dbutils.widgets.text("catalogo_destino", "workspace")
dbutils.widgets.text("schema_bronze", "bronze_app")
dbutils.widgets.text("schema_silver", "silver_app")
dbutils.widgets.text("minutos_offline", "10")
dbutils.widgets.dropdown(
    "tabela_amostra",
    "estado_camara_atual",
    [
        "cd", "usuario", "termometro", "camara_frigorifica",
        "leitura_temperatura", "alerta", "atendimento", "justificativa",
        "categoria", "lote", "lote_camara_frigorifica", "estado_camara_atual",
    ],
)

CATALOGO_DESTINO = dbutils.widgets.get("catalogo_destino").strip()
SCHEMA_BRONZE = dbutils.widgets.get("schema_bronze").strip()
SCHEMA_SILVER = dbutils.widgets.get("schema_silver").strip()
TABELA_AMOSTRA = dbutils.widgets.get("tabela_amostra").strip()

try:
    MINUTOS_OFFLINE = int(dbutils.widgets.get("minutos_offline").strip())
except ValueError as erro:
    raise ValueError("minutos_offline deve ser um número inteiro positivo.") from erro


def validar_identificador(valor: str, nome: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", valor):
        raise ValueError(f"Identificador inválido em {nome}: {valor!r}")
    return valor


CATALOGO_DESTINO = validar_identificador(CATALOGO_DESTINO, "catalogo_destino")
SCHEMA_BRONZE = validar_identificador(SCHEMA_BRONZE, "schema_bronze")
SCHEMA_SILVER = validar_identificador(SCHEMA_SILVER, "schema_silver")

if MINUTOS_OFFLINE <= 0:
    raise ValueError("minutos_offline deve ser maior que zero.")

BRONZE = f"{CATALOGO_DESTINO}.{SCHEMA_BRONZE}"
SILVER = f"{CATALOGO_DESTINO}.{SCHEMA_SILVER}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")


def ler_bronze(nome_tabela: str) -> DataFrame:
    nome_tabela = validar_identificador(nome_tabela, "nome_tabela")
    tabela_completa = f"{BRONZE}.bronze_app_{nome_tabela}"
    if not spark.catalog.tableExists(tabela_completa):
        raise ValueError(f"Tabela Bronze não encontrada: {tabela_completa}")
    return spark.table(tabela_completa)


def texto_normalizado(coluna: str):
    return F.upper(F.trim(F.col(coluna).cast("string")))


def metadado_bronze():
    return F.col("_bronze_ingested_at").cast("timestamp").alias("_bronze_ingested_at")


def gravar_silver(df: DataFrame, nome_tabela: str) -> None:
    nome_tabela = validar_identificador(nome_tabela, "nome_tabela")
    tabela_completa = f"{SILVER}.silver_app_{nome_tabela}"
    (
        df.withColumn("_silver_transformed_at", F.current_timestamp())
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_completa)
    )
    print(f"Gravada: {tabela_completa}")


In [ ]:
# Tabelas de referência: uma linha por entidade operacional.
cd = (
    ler_bronze("tb_cd").dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id_cd"),
        F.trim(F.col("nome").cast("string")).alias("nome_cd"),
        metadado_bronze(),
    )
)

usuario = (
    ler_bronze("tb_usuario").dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id_usuario"),
        F.trim(F.col("nome").cast("string")).alias("nome_usuario"),
        texto_normalizado("nivel_acesso").alias("nivel_acesso"),
        F.col("cod_cd").cast("long").alias("id_cd"),
        F.col("cod_gestor").cast("long").alias("id_gestor"),
        metadado_bronze(),
    )
)

termometro = (
    ler_bronze("tb_termometro").dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id_termometro"),
        F.trim(F.col("modelo").cast("string")).alias("modelo_termometro"),
        metadado_bronze(),
    )
)

camara_frigorifica = (
    ler_bronze("tb_camara_frigorifica").dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id_camara"),
        F.trim(F.col("modelo").cast("string")).alias("modelo_camara"),
        F.trim(F.col("localizacao").cast("string")).alias("localizacao"),
        F.col("temperatura_min").cast("decimal(5,2)").alias("temperatura_min"),
        F.col("temperatura_max").cast("decimal(5,2)").alias("temperatura_max"),
        F.col("cod_cd").cast("long").alias("id_cd"),
        F.col("cod_termometro").cast("long").alias("id_termometro"),
        metadado_bronze(),
    )
)

# Fatos: uma linha por leitura, alerta, atendimento ou justificativa.
leitura_temperatura = (
    ler_bronze("tb_leitura_temperatura").dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id_leitura"),
        F.col("cod_termometro").cast("long").alias("id_termometro"),
        F.col("cod_alerta").cast("long").alias("id_alerta"),
        F.col("temperatura").cast("decimal(5,2)").alias("temperatura"),
        F.col("data_hora").cast("timestamp").alias("data_hora_leitura"),
        F.trim(F.col("id_evento_redis").cast("string")).alias("id_evento_redis"),
        metadado_bronze(),
    )
)

alerta = (
    ler_bronze("tb_alerta").dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id_alerta"),
        F.col("cod_camara_frigorifica").cast("long").alias("id_camara"),
        # Intervalos são preservados como texto nesta versão. A conversão para
        # minutos/segundos será definida junto com a regra formal de SLA.
        F.col("vida_util_referencia_horas").cast("string").alias("vida_util_referencia_horas"),
        texto_normalizado("nivel_atual").alias("nivel_atual"),
        F.col("data_hora").cast("timestamp").alias("data_hora_alerta"),
        texto_normalizado("tipo").alias("tipo_alerta"),
        # Preserva cada gravidade como categoria individual; não agrupa nem
        # valida uma enumeração até o grupo confirmar os valores oficiais.
        texto_normalizado("nivel_gravidade").alias("nivel_gravidade"),
        texto_normalizado("status").alias("status_alerta"),
        metadado_bronze(),
    )
)

atendimento = (
    ler_bronze("tb_atendimento").dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id_atendimento"),
        F.col("cod_alerta").cast("long").alias("id_alerta"),
        F.col("cod_usuario").cast("long").alias("id_usuario_responsavel"),
        F.col("data_hora_reconhecimento").cast("timestamp").alias("data_hora_reconhecimento"),
        F.col("data_hora_resolucao").cast("timestamp").alias("data_hora_resolucao"),
        texto_normalizado("status").alias("status_atendimento"),
        metadado_bronze(),
    )
)

justificativa = (
    ler_bronze("tb_justificativa").dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id_justificativa"),
        F.col("cod_atendimento").cast("long").alias("id_atendimento"),
        F.trim(F.col("motivo").cast("string")).alias("motivo"),
        F.col("descricao").cast("string").alias("descricao"),
        F.col("data_hora").cast("timestamp").alias("data_hora_justificativa"),
        metadado_bronze(),
    )
)

# Referências de lote/categoria são mantidas para análises futuras de vida útil.
# Elas não substituem o valor de referência capturado no próprio alerta.
categoria = (
    ler_bronze("tb_categoria").dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id_categoria"),
        F.trim(F.col("nome").cast("string")).alias("nome_categoria"),
        F.col("temperatura_min").cast("decimal(5,2)").alias("temperatura_min"),
        F.col("temperatura_max").cast("decimal(5,2)").alias("temperatura_max"),
        F.col("vida_util_horas").cast("string").alias("vida_util_horas"),
        metadado_bronze(),
    )
)

lote = (
    ler_bronze("tb_lote").dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id_lote"),
        F.trim(F.col("codigo_lote").cast("string")).alias("codigo_lote"),
        F.col("cod_categoria").cast("long").alias("id_categoria"),
        F.col("data_fabricacao").cast("date").alias("data_fabricacao"),
        F.col("data_validade").cast("date").alias("data_validade"),
        texto_normalizado("status").alias("status_lote"),
        metadado_bronze(),
    )
)

lote_camara_frigorifica = (
    ler_bronze("tb_lote_camara_frigorifica").dropDuplicates(["id"])
    .select(
        F.col("id").cast("long").alias("id_movimentacao"),
        F.col("cod_lote").cast("long").alias("id_lote"),
        F.col("cod_camara_frigorifica").cast("long").alias("id_camara"),
        F.col("data_entrada").cast("timestamp").alias("data_entrada"),
        F.col("data_saida").cast("timestamp").alias("data_saida"),
        metadado_bronze(),
    )
)


In [ ]:
def validar_sem_registros(df: DataFrame, regra: str) -> None:
    if df.limit(1).count() > 0:
        raise ValueError(f"Validação da Silver falhou: {regra}")


def validar_fk(df: DataFrame, coluna_fk: str, dimensao: DataFrame, coluna_pk: str, regra: str, aceita_nulo: bool = False) -> None:
    sem_referencia = df.join(
        dimensao.select(F.col(coluna_pk).alias("_id_referencia")),
        df[coluna_fk] == F.col("_id_referencia"),
        "left_anti",
    )
    if aceita_nulo:
        sem_referencia = sem_referencia.filter(F.col(coluna_fk).isNotNull())
    validar_sem_registros(sem_referencia, regra)


# Chaves primárias devem continuar válidas após os casts.
ENTIDADES_COM_ID = {
    "cd": (cd, "id_cd"),
    "usuario": (usuario, "id_usuario"),
    "termometro": (termometro, "id_termometro"),
    "camara_frigorifica": (camara_frigorifica, "id_camara"),
    "leitura_temperatura": (leitura_temperatura, "id_leitura"),
    "alerta": (alerta, "id_alerta"),
    "atendimento": (atendimento, "id_atendimento"),
    "justificativa": (justificativa, "id_justificativa"),
    "categoria": (categoria, "id_categoria"),
    "lote": (lote, "id_lote"),
    "lote_camara_frigorifica": (lote_camara_frigorifica, "id_movimentacao"),
}

for nome, (df, coluna_id) in ENTIDADES_COM_ID.items():
    validar_sem_registros(df.filter(F.col(coluna_id).isNull()), f"{nome}: chave primária nula ou inválida")

validar_sem_registros(
    camara_frigorifica.filter(
        F.col("temperatura_min").isNull()
        | F.col("temperatura_max").isNull()
        | (F.col("temperatura_min") >= F.col("temperatura_max"))
    ),
    "câmara com faixa de temperatura ausente ou inválida",
)
validar_sem_registros(
    alerta.filter(
        F.col("status_alerta").isNull()
        | ~F.col("status_alerta").isin("ATIVO", "RECONHECIDO", "RESOLVIDO")
    ),
    "status de alerta fora do domínio operacional conhecido",
)
validar_sem_registros(
    atendimento.filter(
        F.col("status_atendimento").isNull()
        | ~F.col("status_atendimento").isin("PENDENTE", "EM_ANDAMENTO", "RESOLVIDO")
    ),
    "status de atendimento fora do domínio conhecido",
)

# Relacionamentos exigidos pelo modelo operacional.
validar_fk(camara_frigorifica, "id_cd", cd, "id_cd", "câmara sem CD correspondente")
validar_fk(camara_frigorifica, "id_termometro", termometro, "id_termometro", "câmara sem termômetro correspondente")
validar_fk(usuario.filter(F.col("id_cd").isNotNull()), "id_cd", cd, "id_cd", "usuario com CD inexistente")
validar_fk(usuario, "id_gestor", usuario, "id_usuario", "gestor inexistente", aceita_nulo=True)
validar_fk(alerta, "id_camara", camara_frigorifica, "id_camara", "alerta sem câmara correspondente")
validar_fk(leitura_temperatura, "id_termometro", termometro, "id_termometro", "leitura sem termômetro correspondente")
validar_fk(leitura_temperatura, "id_alerta", alerta, "id_alerta", "leitura com alerta inexistente", aceita_nulo=True)
validar_fk(atendimento, "id_alerta", alerta, "id_alerta", "atendimento sem alerta correspondente")
validar_fk(atendimento, "id_usuario_responsavel", usuario, "id_usuario", "atendimento com usuário inexistente", aceita_nulo=True)
validar_fk(justificativa, "id_atendimento", atendimento, "id_atendimento", "justificativa sem atendimento correspondente")
validar_fk(lote, "id_categoria", categoria, "id_categoria", "lote sem categoria correspondente")
validar_fk(lote_camara_frigorifica, "id_lote", lote, "id_lote", "movimentação sem lote correspondente")
validar_fk(lote_camara_frigorifica, "id_camara", camara_frigorifica, "id_camara", "movimentação sem câmara correspondente")

# O banco permite somente um alerta ATIVO/RECONHECIDO por câmara.
alertas_abertos = alerta.filter(F.col("status_alerta").isin("ATIVO", "RECONHECIDO"))
validar_sem_registros(
    alertas_abertos.groupBy("id_camara").count().filter(F.col("count") > 1),
    "mais de um alerta aberto para a mesma câmara",
)

# Última leitura por termômetro determina o estado térmico atual da câmara.
janela_ultima_leitura = Window.partitionBy("id_termometro").orderBy(
    F.col("data_hora_leitura").desc_nulls_last(),
    F.col("id_leitura").desc_nulls_last(),
)
ultima_leitura = (
    leitura_temperatura
    .filter(F.col("data_hora_leitura").isNotNull())
    .withColumn("_ordem_leitura", F.row_number().over(janela_ultima_leitura))
    .filter(F.col("_ordem_leitura") == 1)
    .select(
        "id_termometro",
        F.col("id_leitura").alias("id_ultima_leitura"),
        F.col("temperatura").alias("temperatura_atual"),
        F.col("data_hora_leitura").alias("ultima_leitura_em"),
    )
)

alerta_aberto_por_camara = alertas_abertos.select(
    "id_camara",
    F.col("id_alerta").alias("id_alerta_aberto"),
    F.col("nivel_gravidade").alias("gravidade_alerta_atual"),
    F.col("status_alerta").alias("status_alerta_atual"),
    F.col("nivel_atual").alias("nivel_responsavel_atual"),
)

limite_offline = F.expr(f"current_timestamp() - INTERVAL {MINUTOS_OFFLINE} MINUTES")
leitura_ausente = F.col("ultima_leitura_em").isNull()
leitura_antiga = F.col("ultima_leitura_em") <= limite_offline
fora_da_faixa = (
    (F.col("temperatura_atual") < F.col("temperatura_min"))
    | (F.col("temperatura_atual") > F.col("temperatura_max"))
)

estado_camara_atual = (
    camara_frigorifica
    .join(ultima_leitura, "id_termometro", "left")
    .join(alerta_aberto_por_camara, "id_camara", "left")
    .withColumn(
        "estado_conectividade",
        F.when(leitura_ausente, F.lit("SEM_DADOS"))
        .when(leitura_antiga, F.lit("OFFLINE"))
        .otherwise(F.lit("ONLINE")),
    )
    .withColumn(
        "temperatura_fora_da_faixa",
        F.when(leitura_ausente | leitura_antiga, F.lit(None).cast("boolean"))
        .otherwise(fora_da_faixa),
    )
    .withColumn(
        "estado_termico",
        F.when(leitura_ausente, F.lit("SEM_DADOS"))
        .when(leitura_antiga, F.lit("OFFLINE"))
        .when(fora_da_faixa, F.lit("FORA_DA_FAIXA"))
        .otherwise(F.lit("NORMAL")),
    )
    .withColumn("minutos_offline_configurados", F.lit(MINUTOS_OFFLINE))
)


In [ ]:
# Todas as validações acontecem antes de substituir qualquer tabela Silver.
TABELAS_SILVER = {
    "cd": cd,
    "usuario": usuario,
    "termometro": termometro,
    "camara_frigorifica": camara_frigorifica,
    "leitura_temperatura": leitura_temperatura,
    "alerta": alerta,
    "atendimento": atendimento,
    "justificativa": justificativa,
    "categoria": categoria,
    "lote": lote,
    "lote_camara_frigorifica": lote_camara_frigorifica,
    "estado_camara_atual": estado_camara_atual,
}

for nome, df in TABELAS_SILVER.items():
    gravar_silver(df, nome)

print(f"Silver concluída: {len(TABELAS_SILVER)} tabelas em {SILVER}")


In [ ]:
TABELAS_AMOSTRA = set(TABELAS_SILVER)
if TABELA_AMOSTRA not in TABELAS_AMOSTRA:
    raise ValueError(f"tabela_amostra inválida: {TABELA_AMOSTRA}")

nome_completo = f"{SILVER}.silver_app_{TABELA_AMOSTRA}"
df_amostra = spark.table(nome_completo)
print(f"Amostra: {nome_completo}")
df_amostra.printSchema()
display(df_amostra.limit(20))
